In [1]:
# === Imports ===

from typing import Tuple

import numpy as np
from numba import jit
from matplotlib import pyplot as plt
import matplotlib.style as mplstyle

mplstyle.use("./docs/pyscopee.mplstyle")

%matplotlib widget

# === Constants ===

DECIMATION_FACTOR = 10
SPLINE_POLY_DEGREE = 3  # or 1
ZOOM_IN_INDICES = slice(5000, 6000)

# === Main ===

# the regularly sampled signal is loaded
data_regular = np.loadtxt("signal_regular.txt", delimiter=",", skiprows=1)

t_values_regular = data_regular[:, 0]
y_values_regular = data_regular[:, 1]

# the irregularly sampled signal is loaded
data_irregular = np.loadtxt("signal_irregular_with_noise.txt", delimiter=",", skiprows=1)

t_values_irregular = data_irregular[:, 0]
y_values_irregular = data_irregular[:, 2]
noise_stddevs_irregular = data_irregular[:, 3]

In [ ]:
freqs = np.fft.rfftfreq(
    len(y_values_regular), d=t_values_regular[1] - t_values_regular[0]
)
signal_fft = np.fft.rfft(y_values_regular)

sinc_signal = np.sinc(2.0 * 20_000.0 * t_values_regular)
sinc_fft = np.fft.rfft(sinc_signal)

plt.close("all")

fig, ax = plt.subplots()

ax.plot(
    freqs, np.abs(signal_fft) / np.abs(signal_fft).max(), label="FFT of regular signal"
)
ax.plot(freqs, np.abs(sinc_fft) / np.abs(sinc_fft).max(), label="FFT of sinc signal")

In [ ]:
test = np.random.rand(50_000)
indices = np.arange(500, 550)

%timeit test[indices]
%timeit test[500:550]

In [ ]:
def generate_kernel_matrix_specs(
    t_values: np.ndarray,
    t_grid: np.ndarray,
    window_size: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Generates the specifications for the generation of the transposed sinc kernel matrix
    ``A.T`` to compute the matrix product ``A.T @ W @ A`` without forming ``A.T``
    explicitly.
    For this, it finds the distances of between all the grid points ``t_grid`` and the
    values in ``t_values`` if they are not more than ``window_size`` apart.
    Besides, it also evaluates the covariance structure between the grid points which
    is basically the sparsity pattern of ``A.T @ W @ A``.

    Parameters
    ----------
    t_values : :class:`numpy.ndarray` of shape ``(n,)``
        The irregularly sampled time values.
    t_grid : :class:`numpy.ndarray` of shape ``(m,)``
        The regularly sampled grid points.
    window_size : :class:`float`
        The size of the window around each grid point to consider.

    Returns
    -------
    nonzero_covariance_indices : :class:`numpy.ndarray` of shape ``(k, 2)``
        The indices of the grid points between which the covariance is non-zero.
        Its ``i``-th row contains the indices of ``t_grid`` that have non-zero
        covariance with ``t_grid[i]``.
    distances : :class:`numpy.ndarray` of shape ``(l,)``
        The distances of the values in ``t_values`` to the grid points in ``t_grid``
        that are within the window size. It is a compressed version of the matrix
        ``D.T`` from which the transposed sinc kernel matrix ``A.T`` can be computed.
        Please refer to the Notes section for details.
    indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point. Its ``i``-th row contain the indices
        of the first and last value in ``t_values`` that are within the window size
        around ``t_grid[i]``.
        Please refer to the Notes section for details.
    indptr: :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``distances`` and ``indices`` arrays that determine
        how to unpack the distances and indices for each grid point. Its ``i``-th and
        ``i+1``-th element contain the start and stop index of the ``distances``
        associated with ``t_grid[i]``.
        Please refer to the Notes section for details.

    Notes
    -----
    ``distances``, ``indices``, and ``indptr`` are almost equivalent to the ``data``,
    ``indices``, and ``indptr`` arrays in the sparse CSR matrix format. The only
    difference is that ``indices`` is not a 1D-vector full of individual indices, but
    a 2D-Array where each row contains the indices of the first and last value to store
    in the specific row of the matrix to exploit the fact that these indices are always
    consecutive.

    So to unpack the matrix ``D.T``, the following code can be used:

    ```python
    matrix_D = np.zeros(shape=(len(t_grid), len(t_values)))
    for grid_index, (index_from, index_to) in enumerate(indices):
        matrix_D[grid_index, index_from:index_to] = distances[indptr[grid_index]:indptr[grid_index + 1]]
    ```

    """  # noqa: E501

    # the range around each of the values in ``t_grid`` is calculated
    # NOTE: they need to be clipped to the minimum and maximum grid values
    t_grid_min, t_grid_max = t_grid[0], t_grid[-1]
    half_window_size = window_size / 2.0
    t_grid_range_lower = np.maximum(t_grid - half_window_size, t_grid_min)
    t_grid_range_upper = np.minimum(t_grid + half_window_size, t_grid_max)

    # afterwards, the covariance structure between the grid points is calculated, i.e.,
    # the indices at which the matrix ``A.T @ W @ A`` will have non-zero entries
    indices_from = np.searchsorted(t_grid, t_grid_range_lower, side="left")
    indices_to = np.searchsorted(t_grid, t_grid_range_upper, side="right")
    nonzero_covariance_indices = np.concatenate(
        (
            indices_from.reshape((-1, 1)),
            indices_to.reshape((-1, 1)),
        ),
        axis=1,
    )

    # now, all the ``t_values``that are within the range of a grid point are found
    indices_from = np.searchsorted(t_values, t_grid_range_lower, side="left")
    indices_to = np.searchsorted(t_values, t_grid_range_upper, side="right")
    indptr = np.concatenate(
        (
            np.array([0], dtype=indices_from.dtype),
            np.cumsum(indices_to - indices_from)
        ),
    )

    # the distances to the grid points are calculated
    distances = np.empty(shape=(indptr[-1]), dtype=t_values.dtype)
    write_index_to = 0

    for grid_index, grid_value in enumerate(t_grid):
        read_index_from = indices_from[grid_index]
        read_index_to = indices_to[grid_index]
        write_index_from = write_index_to
        write_index_to = indptr[grid_index + 1]

        distances[write_index_from:write_index_to] = t_values[read_index_from:read_index_to] - grid_value

    return (
        nonzero_covariance_indices,
        distances,
        np.concatenate(
            (
                indices_from.reshape((-1, 1)),
                indices_to.reshape((-1, 1)),
            ),
            axis=1,
        ),
        indptr,
    )

@jit(nopython=True)
def sinh_zeromapped_window(x: np.ndarray, exponent: float) -> np.ndarray:
    return np.power(
         np.sinh(1.0 - np.square(x)) / np.sinh(1.0),
         exponent,
     )
    one_minus_x_squared = 1.0 - np.square(x)
    return np.power(
        one_minus_x_squared - (1.0/6.0) * one_minus_x_squared * one_minus_x_squared * one_minus_x_squared,
        exponent,
    )

def promote_distance_matrix_to_sinc_kernel_matrix(
    distances: np.ndarray,
    bandlimit_frequency: float,
    exponent: float,
    window_size: float,
) -> np.ndarray:
    """
    Promotes the distance matrix to a sinc kernel matrix by computing the sinc kernel
    values corresponding to the distances and multiplying them with the window values.

    The function ``generate_kernel_matrix_specs`` already limited the distances to
    compute only those that are within the window size around the grid points, so the
    sinc and the window function do not need to be zeroed out here and only nonzero
    entries will be computed.

    """

    return np.sinc(2.0 * bandlimit_frequency * distances) * sinh_zeromapped_window(
        x=(0.5 / window_size) * distances, exponent=exponent,)

def apply_weights_to_sinc_kernel_matrix(
    sinc_kernel_values: np.ndarray,
    t_value_indices: np.ndarray,
    t_value_indptr: np.ndarray,
    sqrt_weights: np.ndarray,
) -> np.ndarray:
    """
    Appplies the square root of the weights to the sinc kernel values to pre-compute
    the weighted kernel matrix ``A.T @ sqrt(W)``.

    For the arrangement of ``sinc_kernel_values``, ``t_value_indices``, and
    ``t_value_indptr``, please refer to the documentation of the function
    :func:`generate_kernel_matrix_specs`.

    Parameters
    ----------
    sinc_kernel_values : :class:`numpy.ndarray` of shape ``(l,)``
        The values of the sinc kernel function evaluated at the distances between the
        grid points and the values in ``t_values``.
    t_value_indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point.
    t_value_indptr : :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``sinc_kernel_values`` array that determine how to
        unpack the values for each grid point.
    sqrt_weights : :class:`numpy.ndarray` of shape ``(n,)``
        The square root of the weights to apply to the values in ``t_values``. Its
        ``j``-th element contains the square root of the weight to apply to
        ``t_values[j]``.

    Returns
    -------
    weighted_sinc_kernel_matrix : :class:`numpy.ndarray` of shape ``(l,)``
        The weighted sinc kernel matrix ``A.T @ sqrt(W)`` that is still stored in the
        same compressed format as ``sinc_kernel_values``.

    """ # noqa: E501

    weighted_sinc_kernel_matrix = np.empty_like(sinc_kernel_values)

    for grid_index, (index_from, index_to) in enumerate(t_value_indices):
        data_index_from = t_value_indptr[grid_index]
        data_index_to = t_value_indptr[grid_index + 1]

        weighted_sinc_kernel_matrix[data_index_from:data_index_to] = (
            sinc_kernel_values[data_index_from:data_index_to] * sqrt_weights[index_from:index_to]
        )

    return weighted_sinc_kernel_matrix




find_grid_distances_in_window_reach_jit = jit(generate_kernel_matrix_specs, nopython=True)
promote_distance_matrix_to_sinc_kernel_matrix_jit = jit(promote_distance_matrix_to_sinc_kernel_matrix, nopython=True)
apply_weights_to_sinc_kernel_matrix_jit = jit(apply_weights_to_sinc_kernel_matrix, nopython=True)

a = np.linspace(0, 10, 50_000)
b = np.linspace(0, 10, 5_000)
print("delta b", b[1] - b[0])

_, dist, indices, indptr = generate_kernel_matrix_specs(a, b, 5 * (b[1] - b[0]))
new_dist = promote_distance_matrix_to_sinc_kernel_matrix(dist, 2_000, 5.0, 5 * (b[1] - b[0]))

wghts = np.ones_like(a)
%timeit _, dist, indices, indptr = generate_kernel_matrix_specs(a, b, 5 * (b[1] - b[0]))
%timeit _, dist, indices, indptr = find_grid_distances_in_window_reach_jit(a, b, 5 * (b[1] - b[0]))
%timeit new_dist = promote_distance_matrix_to_sinc_kernel_matrix(dist, 2_000, 5.0, 5 * (b[1] - b[0]))
%timeit new_dist = promote_distance_matrix_to_sinc_kernel_matrix_jit(dist, 2_000, 5.0, 5 * (b[1] - b[0]))
%timeit weighted_dist = apply_weights_to_sinc_kernel_matrix(new_dist, indices, indptr, wghts)
%timeit weighted_dist = apply_weights_to_sinc_kernel_matrix_jit(new_dist, indices, indptr, wghts)


""" matrix_D = np.zeros(shape=(len(b), len(a)))
for grid_index, (index_from, index_to) in enumerate(indices):
    matrix_D[grid_index, index_from:index_to] = dist[indptr[grid_index]:indptr[grid_index + 1]]

fig, ax = plt.subplots()

ax.imshow(matrix_D, aspect="auto", cmap="viridis") """


# print(generate_kernel_matrix_specs(a, b, 0.1))
# print(find_grid_distances_in_window_reach_jit(a, b, 0.1))

# %timeit find_grid_distances_in_window_reach(a, b, 0.1)
# %timeit find_grid_distances_in_window_reach_jit(a, b, 0.1)

In [ ]:
from numba import jit


""" def sinh_zeromapped_window(x: np.ndarray, exponent: float) -> np.ndarray:
    return np.power(
        np.sinh(1.0 - np.square(x)) / np.sinh(1.0),
        exponent,
    ) """


def windowed_sinc_kernel(
    x: np.ndarray, bandlimit_freq: float, window_width: float, window_exponent: float
) -> np.ndarray:
    return np.sinc(
        2.0 * bandlimit_freq * x
    )  # * sinh_zeromapped_window(x, window_width, window_exponent)


plt.close("all")

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

bandlimit_freq = 20_000.0
exponent = 5.0

# y_values = sinh_zeromapped_window(t_values_regular exponent)
y_values2 = windowed_sinc_kernel(t_values_regular, bandlimit_freq, 0.05, exponent)

freqs = np.fft.rfftfreq(len(y_values), d=t_values_regular[1] - t_values_regular[0])
sinc_window_fft = np.fft.rfft(y_values2)

# ax.plot(t_values_regular, y_values)
ax.plot(t_values_regular, y_values2)

ax2.plot(freqs, np.abs(sinc_window_fft) / np.abs(sinc_window_fft).max())

plt.show()